In [5]:
import torch
from torch import nn
from torch.ao import quantization
from torchvision import models
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder


class QuantizedVGG16(nn.Module):
    def __init__(self, model_fp32):
        super(QuantizedVGG16, self).__init__()
        self.quant = quantization.QuantStub()
        self.dequant = quantization.DeQuantStub()
        self.model_fp32 = model_fp32
    
    def forward(self, x):
        x = self.quant(x)
        x = self.model_fp32(x)
        x = self.dequant(x)
        return x


hyperparams = {
    "batch_size": 4,
    "learning_rate": 0.0001,
    "epochs": 5,
    "transform": transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.48235, 0.45882, 0.40784],
                std=[0.229, 0.224, 0.225],
            ),
        ]
    ),
}

# -----------------------------
# FP32 모델 로드
# -----------------------------
model = models.vgg16(num_classes=2)
model.load_state_dict(torch.load("../models/VGG16.pt"))

# quantization은 CPU에서 수행
model.eval()
model.to("cpu")

# -----------------------------
# Quantization Wrapper
# -----------------------------
quantized_model = QuantizedVGG16(model)
quantized_model.eval()

# -----------------------------
# Quantization Backend 설정
# -----------------------------
# quantization_backend = "fbgemm"  # x86 CPU에서 최적화된 백엔드
quantization_backend = "qnnpack"  # ARM CPU에서 최적화된 백엔드
torch.backends.quantized.engine = quantization_backend

quantized_model.qconfig = quantization.get_default_qconfig(quantization_backend)

# -----------------------------
# Prepare (Observer 삽입)
# -----------------------------
model_static_quantized = quantization.prepare(quantized_model, inplace=False)

# -----------------------------
# Calibration Dataset
# -----------------------------
calibration_dataset = ImageFolder(
    "../datasets/pet/test",
    transform=hyperparams["transform"]
)

calibration_dataloader = DataLoader(
    calibration_dataset,
    batch_size=hyperparams["batch_size"]
)

# -----------------------------
# Calibration
# -----------------------------
with torch.no_grad():
    for i, (images, target) in enumerate(calibration_dataloader):
        if i >= 10:
            break
        model_static_quantized(images)

# -----------------------------
# Convert (INT8 변환)
# -----------------------------
model_static_quantized = quantization.convert(model_static_quantized, inplace=False)

# -----------------------------
# TorchScript 저장
# -----------------------------
script_model = torch.jit.script(model_static_quantized)
torch.jit.save(script_model, "../models/PTSQ_VGG16.pt")